# Prediction Curve Comparison

Plot true values, DARNet, and all selected baselines from `draw/**/test_agg.csv`.


In [ ]:

from pathlib import Path
from collections import defaultdict, OrderedDict
import csv
import math
import re

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

ROOT = Path.cwd().parent if Path.cwd().name == 'PredictionPlot' else Path.cwd()
DRAW_DIR = ROOT / 'draw'
OUT_DIR = ROOT / 'PredictionPlot'
FIG_DIR = OUT_DIR / 'figures'
DATA_DIR = OUT_DIR / 'plot_data'
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

DATASETS = ['Abilene', 'Geant', 'Seattle']
PRED_LENS = [5, 10, 15, 20]
D_MODEL = 256
TARGET_COL = 'TC0'

MODEL_DIR_TO_LABEL = OrderedDict([
    ('net', 'DARNet'),
    ('PMDformer', 'PMDformer'),
    ('iTransformer', 'iTransformer'),
    ('FEDformer', 'FEDformer'),
    ('FeTS', 'FeTS'),
    ('HMformer', 'HMformer'),
    ('PatchTST', 'PatchTST'),
    ('timesnet', 'TimesNet'),
    ('WPMixer', 'WPMixer'),
    ('P_sLSTM', 'P_sLSTM'),
    ('xLSTMTime', 'xLSTMTime'),
    ('xlstm_mixer', 'xLSTM-Mixer'),
])
MODEL_ORDER = list(MODEL_DIR_TO_LABEL.values())
DISPLAY_ORDER = ['True'] + MODEL_ORDER

LINE_STYLE = {
    'True': dict(color='black', linewidth=2.6, linestyle='-', alpha=0.95, zorder=20),
    'DARNet': dict(color='#d62728', linewidth=2.2, linestyle='-', alpha=0.95, zorder=18),
    'PMDformer': dict(color='#1f77b4', linewidth=1.25, linestyle='-', alpha=0.86),
    'iTransformer': dict(color='#ff7f0e', linewidth=1.25, linestyle='-', alpha=0.86),
    'FEDformer': dict(color='#2ca02c', linewidth=1.25, linestyle='-', alpha=0.86),
    'FeTS': dict(color='#9467bd', linewidth=1.25, linestyle='-', alpha=0.86),
    'HMformer': dict(color='#8c564b', linewidth=1.25, linestyle='-', alpha=0.86),
    'PatchTST': dict(color='#e377c2', linewidth=1.25, linestyle='-', alpha=0.86),
    'TimesNet': dict(color='#7f7f7f', linewidth=1.25, linestyle='-', alpha=0.86),
    'WPMixer': dict(color='#bcbd22', linewidth=1.25, linestyle='-', alpha=0.86),
    'P_sLSTM': dict(color='#17becf', linewidth=1.25, linestyle='-', alpha=0.86),
    'xLSTMTime': dict(color='#00429d', linewidth=1.25, linestyle='--', alpha=0.9),
    'xLSTM-Mixer': dict(color='#93003a', linewidth=1.25, linestyle='--', alpha=0.9),
}

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.unicode_minus': False, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.dpi': 180})

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def discover_files():
    records = []
    for p in DRAW_DIR.rglob('test_agg.csv'):
        try:
            dataset, data_tag, model_dir, pl_dm, tc, filename = p.relative_to(DRAW_DIR).parts
        except ValueError:
            continue
        if dataset not in DATASETS or model_dir not in MODEL_DIR_TO_LABEL or tc != TARGET_COL:
            continue
        m = re.fullmatch(r'PL(\d+)_DM(\d+)', pl_dm)
        if not m:
            continue
        pred_len, d_model = int(m.group(1)), int(m.group(2))
        if pred_len not in PRED_LENS or d_model != D_MODEL:
            continue
        records.append({'dataset': dataset, 'data_tag': data_tag, 'model_dir': model_dir, 'model': MODEL_DIR_TO_LABEL[model_dir], 'pred_len': pred_len, 'd_model': d_model, 'target_col': tc, 'path': p})
    return records

def read_curve(path):
    df = pd.read_csv(path)
    df = df[['time_idx', 'true', 'pred']].copy()
    df['time_idx'] = pd.to_numeric(df['time_idx'], errors='coerce').astype('Int64')
    df['true'] = pd.to_numeric(df['true'], errors='coerce')
    df['pred'] = pd.to_numeric(df['pred'], errors='coerce')
    df = df.dropna(subset=['time_idx']).copy()
    df['time_idx'] = df['time_idx'].astype(int)
    return df.sort_values('time_idx').drop_duplicates('time_idx', keep='last')

def build_merged_curve(records_for_combo):
    by_model = {r['model']: r for r in records_for_combo}
    missing_models = [m for m in MODEL_ORDER if m not in by_model]
    if missing_models:
        raise ValueError(f'Missing models: {missing_models}')
    true_sources = []
    merged = None
    for model in MODEL_ORDER:
        df = read_curve(by_model[model]['path'])
        true_sources.append(df[['time_idx', 'true']].rename(columns={'true': f'true_{model}'}))
        pred_df = df[['time_idx', 'pred']].rename(columns={'pred': model})
        merged = pred_df if merged is None else merged.merge(pred_df, on='time_idx', how='inner')
    true_merged = true_sources[0]
    for src in true_sources[1:]:
        true_merged = true_merged.merge(src, on='time_idx', how='inner')
    true_cols = [c for c in true_merged.columns if c.startswith('true_')]
    true_merged['True'] = true_merged[true_cols].median(axis=1, skipna=True)
    true_diff = true_merged[true_cols].max(axis=1, skipna=True) - true_merged[true_cols].min(axis=1, skipna=True)
    max_true_diff = float(np.nanmax(np.abs(true_diff.to_numpy()))) if len(true_diff) else math.nan
    merged = merged.merge(true_merged[['time_idx', 'True']], on='time_idx', how='inner')
    merged = merged[['time_idx', 'True'] + MODEL_ORDER]
    merged = merged.dropna(subset=['True']).sort_values('time_idx').reset_index(drop=True)
    merged.insert(0, 'step', np.arange(len(merged), dtype=int))
    return merged, max_true_diff

def plot_curve(merged, dataset, pred_len, out_pdf, out_png):
    fig, ax = plt.subplots(figsize=(12.5, 5.2))
    x = merged['step'].to_numpy()
    for name in DISPLAY_ORDER:
        if name in merged.columns:
            ax.plot(x, merged[name].to_numpy(), label=name, **LINE_STYLE.get(name, {}))
    ax.set_title(f'{dataset} Prediction Curves (PredLen={pred_len})', fontsize=13, pad=10)
    ax.set_xlabel('Step')
    ax.set_ylabel('Value')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.38)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.legend(ncol=7, loc='upper center', bbox_to_anchor=(0.5, -0.14), frameon=False, fontsize=9, handlelength=2.2, columnspacing=1.0)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(out_pdf, bbox_inches='tight')
    fig.savefig(out_png, bbox_inches='tight', dpi=300)
    plt.close(fig)


In [ ]:

records = discover_files()
print(f'Discovered eligible files: {len(records)}')

with (OUT_DIR / 'discovered_prediction_files.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'data_tag', 'model_dir', 'model', 'pred_len', 'd_model', 'target_col', 'path'])
    writer.writeheader()
    for r in records:
        row = dict(r)
        row['path'] = str(row['path']).replace('\\', '/')
        writer.writerow(row)

by_combo = defaultdict(list)
for r in records:
    by_combo[(r['dataset'], r['pred_len'])].append(r)

index_rows = []
quality_rows = []
for dataset in DATASETS:
    for pred_len in PRED_LENS:
        items = by_combo.get((dataset, pred_len), [])
        present = sorted({r['model'] for r in items})
        missing = [m for m in MODEL_ORDER if m not in present]
        if missing:
            quality_rows.append({'dataset': dataset, 'pred_len': pred_len, 'status': 'missing', 'point_count': 0, 'max_true_diff_across_models': '', 'missing_models': ';'.join(missing)})
            continue
        merged, max_true_diff = build_merged_curve(items)
        data_path = DATA_DIR / f'{safe_name(dataset)}_PL{pred_len}_merged_prediction_curves.csv'
        merged.to_csv(data_path, index=False, encoding='utf-8-sig')
        pdf_path = FIG_DIR / f'{safe_name(dataset)}_PL{pred_len}_prediction_curves.pdf'
        png_path = FIG_DIR / f'{safe_name(dataset)}_PL{pred_len}_prediction_curves.png'
        plot_curve(merged, dataset, pred_len, pdf_path, png_path)
        quality_rows.append({'dataset': dataset, 'pred_len': pred_len, 'status': 'ok', 'point_count': len(merged), 'max_true_diff_across_models': f'{max_true_diff:.12g}', 'missing_models': ''})
        index_rows.append({'dataset': dataset, 'pred_len': pred_len, 'pdf': str(pdf_path.relative_to(OUT_DIR)).replace('\\', '/'), 'png': str(png_path.relative_to(OUT_DIR)).replace('\\', '/'), 'merged_csv': str(data_path.relative_to(OUT_DIR)).replace('\\', '/'), 'point_count': len(merged)})

with (OUT_DIR / 'prediction_figure_index.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'pred_len', 'pdf', 'png', 'merged_csv', 'point_count'])
    writer.writeheader()
    writer.writerows(index_rows)
with (OUT_DIR / 'prediction_curve_quality_check.csv').open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'pred_len', 'status', 'point_count', 'max_true_diff_across_models', 'missing_models'])
    writer.writeheader()
    writer.writerows(quality_rows)

print(f'Generated figures: {len(index_rows)} PDF + {len(index_rows)} PNG')
print(pd.DataFrame(quality_rows))
